# Corrective RAG
### Self-correcting retrieval with confidence-based fallbacks

Corpus: `NIST AI RMF (AI.100-1)` — a 2023 document, so it can't cover anything more recent. That gap is used deliberately below to force the fallback branch.

Unlike Self-RAG, the correction logic here sits **outside** the model, in a separate grading step that runs before generation.

## Step 1: Build the pipeline

In [1]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu pypdf python-dotenv tavily-python -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from tavily import TavilyClient

pages = PyPDFLoader("NIST.AI.100-1.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(pages)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
web_search = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

C:\Users\shiva\AppData\Local\Temp\ipykernel_26388\1099708982.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 48 pages -> 150 chunks -> 150 vectors


## Step 2: Grade retrieval before trusting it
Each retrieved chunk is scored independently; the pattern across all of them decides the confidence outcome.

In [3]:
def grade_chunk(query, chunk):
    prompt = f"""Does the passage below contain information that directly answers the question?
Answer RELEVANT only if it states a specific fact that answers the question.
Answer IRRELEVANT if it merely discusses a related topic without actually answering it -- being on-topic is not enough.

Question: {query}
Passage: {chunk.page_content}
Answer with exactly one word:"""
    label = llm.invoke(prompt).content.strip().upper()
    return "IRRELEVANT" not in label and "RELEVANT" in label  # "IRRELEVANT" contains "RELEVANT" as a substring, so check it first

def decide(query, docs):
    grades = [grade_chunk(query, d) for d in docs]
    if all(grades):
        return "CORRECT", grades
    if not any(grades):
        return "INCORRECT", grades
    return "AMBIGUOUS", grades

## Step 3: Web search fallback
Only called when internal retrieval is graded incorrect or ambiguous.

In [4]:
def search_web(query, max_results=3):
    results = web_search.search(query, max_results=max_results)
    return [f"[Web: {r['title']}] {r['content']}" for r in results.get("results", [])]

## Step 4: The correction pipeline
- **Correct** → keep every chunk as-is.
- **Incorrect** → discard all internal chunks, answer from web search alone.
- **Ambiguous** → keep the internally-graded-relevant chunks and add web search alongside them.

In [5]:
GEN_PROMPT = """Answer the question using only the following context. Note whether each part of your
answer comes from the internal document or the web.

Context:
{context}

Question: {query}
Answer:"""

def corrective_rag(query, k=4):
    docs = vector_store.similarity_search(query, k=k)
    verdict, grades = decide(query, docs)

    if verdict == "CORRECT":
        context_items = [f"[Doc p.{d.metadata['page']}] {d.page_content}" for d in docs]
    elif verdict == "INCORRECT":
        context_items = search_web(query)
    else:  # AMBIGUOUS
        relevant = [f"[Doc p.{d.metadata['page']}] {d.page_content}" for d, g in zip(docs, grades) if g]
        context_items = relevant + search_web(query)

    context = "\n\n".join(context_items)
    answer = llm.invoke(GEN_PROMPT.format(context=context, query=query)).content.strip()
    return verdict, context_items, answer

## Step 5: One query per branch
- A well-covered question -> **Correct**.
- A question about something the 2023 document cannot possibly contain -> **Incorrect**, fully falls back to the web.
- A question half-covered by the document, half not -> **Ambiguous**, blends both sources.

In [6]:
test_queries = [
    ("Correct (expected)", "What is the GOVERN function in the NIST AI Risk Management Framework?"),
    ("Incorrect (expected)", "What new NIST AI guidance or executive orders on AI were published in 2025?"),
    ("Ambiguous (expected)", "How does NIST recommend organizations manage AI risk, and what specific role does the EU AI Act play in that guidance?"),
]

for label, q in test_queries:
    verdict, context_items, answer = corrective_rag(q)
    print(f"--- {label} -> graded {verdict} ---")
    print(f"Query: {q}")
    print(f"Context sources: {len(context_items)} ({sum(1 for c in context_items if c.startswith('[Web'))} from web)")
    print(f"Answer: {answer}\n")

--- Correct (expected) -> graded CORRECT ---
Query: What is the GOVERN function in the NIST AI Risk Management Framework?
Context sources: 4 (0 from web)
Answer: The GOVERN function in the NIST AI Risk Management Framework is a cross-cutting function that is infused throughout AI risk management, enabling other functions of the process. It involves aspects related to compliance and evaluation, which should be integrated into each of the other functions. Strong governance drives and enhances internal practices and norms to facilitate organizational risk culture, with governing authorities determining overarching policies that direct an organization’s mission, goals, values, culture, and risk tolerance. It includes ongoing monitoring, accountability structures, documentation of risks, and processes for engaging with relevant AI actors. 

(Answer derived from the internal document.)



--- Incorrect (expected) -> graded INCORRECT ---
Query: What new NIST AI guidance or executive orders on AI were published in 2025?
Context sources: 3 (3 from web)
Answer: In 2025, the following new NIST AI guidance and executive orders on AI were published:

1. **Executive Order 14179** - Issued on January 23, 2025, this order directed agencies to remove barriers to U.S. AI leadership and revoked prior administration’s AI policies. (Web)
2. **Executive Order 14319** - Signed in July 2025, this order directed the federal government to procure large language models that adhere to "unbiased AI principles." (Web)
3. **Executive Order 14365** - Signed in December 2025, this order aimed to establish a national policy framework for artificial intelligence. (Web)

There is no mention of specific NIST AI guidance published in 2025 in the provided context.



--- Ambiguous (expected) -> graded AMBIGUOUS ---
Query: How does NIST recommend organizations manage AI risk, and what specific role does the EU AI Act play in that guidance?
Context sources: 4 (3 from web)
Answer: NIST recommends organizations manage AI risk through a voluntary, function-based framework consisting of four specific functions: GOVERN, MAP, MEASURE, and MANAGE. These functions help organizations establish a culture of risk management, identify and understand AI risks, assess and track those risks, and prioritize and mitigate them. 

The EU AI Act plays a complementary role by providing a mandatory, risk-tiered regulation that organizations can align with. While NIST's framework is flexible and guidance-based, the EU AI Act imposes specific technical and process requirements for high-risk AI systems, with enforcement penalties for non-compliance. Organizations can use the NIST framework for operational implementation while mapping their controls to the compliance tiers of

## Try it yourself
1. Lower `k` to 2 and see whether that pushes more queries into AMBIGUOUS or INCORRECT — fewer chunks means each one's grade matters more.
2. Feed the grader a query the document covers well but phrase it in language that shares no vocabulary with the text, and see if that alone flips a CORRECT into AMBIGUOUS.
3. Replace `all()`/`any()` with a majority-vote threshold (e.g. >=70% relevant) and compare which queries change branch.